# Apply Frame Classifier

This notebook applies the trained frame classifier to all ADHD/Autism mention contexts and writes a reusable frame-label handoff for downstream LSC notebooks.

In [ ]:
from __future__ import annotations

import hashlib
import json
import pickle
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
MODEL_DIR = PROJECT_ROOT / "data/processed/lsc/classification"
MODEL_PATH = MODEL_DIR / "hierarchical_frame_logistic_models.pkl"
METADATA_PATH = MODEL_DIR / "frame_classifier_metadata.json"
OUTPUT_PATH = MODEL_DIR / "lsc_target_context_frame_labels.csv"
SUMMARY_PATH = MODEL_DIR / "lsc_frame_counts_by_year_unit.csv"


## Load Model and Target Contexts

In [ ]:
if not MODEL_PATH.exists():
    print(f"No trained model found yet: {MODEL_PATH.relative_to(PROJECT_ROOT)}")
    raise SystemExit("Train and validate the classifier first.")

with MODEL_PATH.open("rb") as handle:
    models = pickle.load(handle)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

columns = [
    "doc_id",
    "analysis_unit",
    "raw_form",
    "mention_start_char",
    "mention_end_char",
    "target_sentence_plus_adjacent",
    "lsc_year",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=columns)
target_contexts = contexts.loc[contexts["analysis_unit"].isin(["ADHD", "Autism"])].copy()
def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]

target_contexts = target_contexts.dropna(subset=["target_sentence_plus_adjacent", "lsc_year"])
target_contexts = target_contexts.loc[target_contexts["target_sentence_plus_adjacent"].str.strip().ne("")].copy()
target_contexts["context_id"] = target_contexts.apply(stable_context_id, axis=1)
duplicate_context_ids = target_contexts["context_id"].duplicated().sum()
if duplicate_context_ids:
    raise ValueError(f"Context ID collision or duplicate mention rows found: {duplicate_context_ids}")
print(f"Target contexts to label: {len(target_contexts):,}")

## Embed and Predict

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError as error:
    raise ImportError("Install the msc-nlp environment with sentence-transformers before applying the classifier.") from error


def classifier_text(frame: pd.DataFrame) -> list[str]:
    return [
        f"TARGET={row.analysis_unit}\nPASSAGE={row.target_sentence_plus_adjacent}"
        for row in frame.itertuples(index=False)
    ]


def derive_frame_from_predictions(substantive: bool, clinical: bool | pd.NA, lived: bool | pd.NA) -> str:
    if not substantive:
        return "non_substantive_or_insufficient"
    clinical_bool = bool(clinical)
    lived_bool = bool(lived)
    if clinical_bool and lived_bool:
        return "mixed"
    if clinical_bool:
        return "clinical_only"
    if lived_bool:
        return "lived_only"
    return "substantive_other"

embedder = SentenceTransformer(metadata["embedding_model"])
x_all = embedder.encode(classifier_text(target_contexts), normalize_embeddings=True, show_progress_bar=True)

labels = target_contexts.copy().reset_index(drop=True)
labels["p_substantive"] = models["substantive_target_discourse"].predict_proba(x_all)[:, 1]
labels["p_clinical_given_substantive"] = models["clinical_frame_present"].predict_proba(x_all)[:, 1]
labels["p_lived_given_substantive"] = models["lived_experience_frame_present"].predict_proba(x_all)[:, 1]

labels["predicted_substantive_target_discourse"] = labels["p_substantive"].ge(0.5)
labels["predicted_clinical_frame_present"] = labels["p_clinical_given_substantive"].ge(0.5)
labels["predicted_lived_experience_frame_present"] = labels["p_lived_given_substantive"].ge(0.5)
labels.loc[~labels["predicted_substantive_target_discourse"], "predicted_clinical_frame_present"] = pd.NA
labels.loc[~labels["predicted_substantive_target_discourse"], "predicted_lived_experience_frame_present"] = pd.NA

labels["predicted_derived_frame"] = [
    derive_frame_from_predictions(substantive, clinical, lived)
    for substantive, clinical, lived in zip(
        labels["predicted_substantive_target_discourse"],
        labels["predicted_clinical_frame_present"],
        labels["predicted_lived_experience_frame_present"],
    )
]

labels["w_non_substantive_or_insufficient"] = 1 - labels["p_substantive"]
labels["w_clinical_only"] = labels["p_substantive"] * labels["p_clinical_given_substantive"] * (1 - labels["p_lived_given_substantive"])
labels["w_lived_only"] = labels["p_substantive"] * (1 - labels["p_clinical_given_substantive"]) * labels["p_lived_given_substantive"]
labels["w_mixed"] = labels["p_substantive"] * labels["p_clinical_given_substantive"] * labels["p_lived_given_substantive"]
labels["w_substantive_other"] = labels["p_substantive"] * (1 - labels["p_clinical_given_substantive"]) * (1 - labels["p_lived_given_substantive"])
labels["classifier_version"] = metadata["classifier_version"]
labels["codebook_version"] = metadata["codebook_version"]
labels["prompt_version"] = metadata["prompt_version"]


## Save Frame Labels

In [ ]:
labels.to_csv(OUTPUT_PATH, index=False)
summary = labels.groupby(["lsc_year", "analysis_unit", "predicted_derived_frame"], as_index=False).size().rename(columns={"size": "contexts"})
summary.to_csv(SUMMARY_PATH, index=False)

print("Wrote frame-label outputs:")
for path in [OUTPUT_PATH, SUMMARY_PATH]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

summary.head()
